# LVM-AI – Externe Validatie Notebook

Stap-voor-stap externe validatie van het LVM-AI model op eigen data.

**Drie modellen beschikbaar:**
| Bestand | Input | Output |
|---|---|---|
| `ecg_rest_raw_lvm_symmetric_loss.h5` | ECG only | LV massa (g) |
| `ecg_rest_raw_lvm_asymmetric_loss.h5` | ECG only | LV massa (g) |
| `ecg_rest_raw_age_sex_bmi_lvm_asymmetric_loss.h5` | ECG + leeftijd/geslacht/BMI | LV massa (g) + LVH kans |

**Vereisten:**
```
pip install tensorflow==2.19.0 beautifulsoup4 lxml numpy pandas scikit-learn matplotlib seaborn
```

## Cel 1 – Configuratie: pas hier je paden aan

In [ ]:
import os

# ── Kies het model ────────────────────────────────────────────────────────
MODEL_DIR = "."  # map met de .h5 bestanden

# Kies één van de drie:
MODEL_NAME = "ecg_rest_raw_age_sex_bmi_lvm_asymmetric_loss.h5"   # ECG + leeftijd/geslacht/BMI → LVM + LVH
# MODEL_NAME = "ecg_rest_raw_lvm_asymmetric_loss.h5"             # ECG only → LVM
# MODEL_NAME = "ecg_rest_raw_lvm_symmetric_loss.h5"              # ECG only → LVM

MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME)

# ── ECG-bestanden ─────────────────────────────────────────────────────────
ECG_DIR    = "/pad/naar/mijn/ecg_bestanden"   # map met XML / NPY / CSV
ECG_FORMAT = "xml"                             # "xml" | "npy" | "csv"

# ── Labels ────────────────────────────────────────────────────────────────
LABELS_CSV = "/pad/naar/labels.csv"

# ── Uitvoermap ────────────────────────────────────────────────────────────
OUTPUT_DIR = "./lvm_validation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuratie OK")
print(f"  Model : {MODEL_PATH}")
print(f"  ECGs  : {ECG_DIR}  ({ECG_FORMAT})")
print(f"  Labels: {LABELS_CSV}")

## Cel 2 – Demo: maak synthetische testdata (sla over als je eigen data hebt)

In [ ]:
import numpy as np
import pandas as pd

DEMO_DIR = "./demo_ecgs"
os.makedirs(DEMO_DIR, exist_ok=True)

rng = np.random.default_rng(0)
LEAD_ORDER = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
demo_labels = []

for i in range(20):
    sid = f"patient_{i:03d}"
    # Synthetisch ECG: sinusgolf + ruis, waarden in μV
    t = np.linspace(0, 10, 5000)
    ecg = rng.normal(0, 50, (5000, 12)).astype(np.float32)
    ecg[:, 1] += 200 * np.sin(2 * np.pi * 1.2 * t)  # R-golf in afleid II

    # Opslaan als NPY
    np.save(os.path.join(DEMO_DIR, f"{sid}.npy"), ecg)

    demo_labels.append({
        "sample_id" : sid,
        "lvm_g"     : float(rng.normal(150, 40)),
        "lvh_label" : int(rng.integers(0, 2)),
        "age"       : float(rng.uniform(40, 80)),
        "sex"       : int(rng.integers(0, 2)),
        "bmi"       : float(rng.uniform(20, 35)),
    })

demo_df = pd.DataFrame(demo_labels)
DEMO_LABELS = "./demo_labels.csv"
demo_df.to_csv(DEMO_LABELS, index=False)

# Gebruik demo-paden voor de rest van het notebook
ECG_DIR    = DEMO_DIR
ECG_FORMAT = "npy"
LABELS_CSV = DEMO_LABELS

print(f"Demo-data aangemaakt: {len(demo_labels)} patiënten in {DEMO_DIR}")
demo_df.head()

## Cel 3 – Verwacht formaat van labels.csv

In [ ]:
# Voorbeeld van een correcte labels.csv
voorbeeld = pd.DataFrame([
    {"sample_id": "patient_001", "lvm_g": 145.2, "lvh_label": 0, "age": 58.0, "sex": 1, "bmi": 26.4},
    {"sample_id": "patient_002", "lvm_g": 210.5, "lvh_label": 1, "age": 67.0, "sex": 1, "bmi": 30.1},
    {"sample_id": "patient_003", "lvm_g": 130.0, "lvh_label": 0, "age": 45.0, "sex": 0, "bmi": 23.8},
])
print("Voorbeeld labels.csv:")
print(voorbeeld.to_string(index=False))
print()
print("Kolom-beschrijvingen:")
print("  sample_id  : unieke ID = bestandsnaam zonder extensie")
print("  lvm_g      : CMR-gemeten LV massa in gram (float)")
print("  lvh_label  : 1=LVH, 0=geen LVH (int) – gebruik CMR-drempel of echo")
print("  age        : leeftijd in jaren (float)")
print("  sex        : 0=vrouw, 1=man (int)")
print("  bmi        : BMI in kg/m² (float)")
print()
print("Tip: kolommen lvm_g en lvh_label zijn optioneel (sla op als NaN als niet beschikbaar)")

## Cel 4 – ECG laden en controleren

In [ ]:
import struct, base64, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

LEAD_ORDER   = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
ECG_SAMPLES  = 5000
ECG_NORM     = 2000.0

# ── XML-loader ────────────────────────────────────────────────────────────
def _decode_b64(raw, scale=1.0):
    d = base64.b64decode(raw)
    return np.array([struct.unpack("h", bytes([d[i],d[i+1]]))[0] for i in range(0,len(d),2)], dtype=np.float32) * scale

def load_xml(path):
    import bs4
    with open(path, "r", errors="replace") as f:
        soup = bs4.BeautifulSoup(f, "lxml")
    v = {}
    for wf in soup.find_all("waveform"):
        wt = wf.find("waveformtype")
        if wt is None or wt.text.strip() != "Rhythm":
            continue
        for ld in wf.find_all("leaddata"):
            lid   = ld.find("leadid").text.strip()
            sc    = float(ld.find("leadamplitudeunitsperbit").text) if ld.find("leadamplitudeunitsperbit") else 1.0
            v[lid] = _decode_b64(ld.find("waveformdata").text.strip(), sc)
        break
    v["III"] = v["II"] - v["I"]
    v["aVR"] = -0.5*(v["I"]+v["II"])
    v["aVL"] = v["I"] - v["II"]/2
    v["aVF"] = v["II"] - v["I"]/2
    def rs(x):
        n=len(x); return x if n==ECG_SAMPLES else np.interp(np.linspace(0,n,ECG_SAMPLES),np.arange(n),x)
    return np.column_stack([rs(v[l]) for l in LEAD_ORDER]).astype(np.float32)

# ── NPY-loader ────────────────────────────────────────────────────────────
def load_npy(path):
    a = np.load(path).astype(np.float32)
    return a.T if a.shape == (12, ECG_SAMPLES) else a

# ── CSV-loader ────────────────────────────────────────────────────────────
def load_csv_ecg(path):
    return pd.read_csv(path)[LEAD_ORDER].values.astype(np.float32)

LOADERS = {"xml": load_xml, "npy": load_npy, "csv": load_csv_ecg}

# ── Test: laad één ECG ────────────────────────────────────────────────────
sample_files = sorted(Path(ECG_DIR).glob(f"*.{ECG_FORMAT}"))
if not sample_files:
    sample_files = sorted(Path(ECG_DIR).iterdir())

test_ecg = LOADERS[ECG_FORMAT](str(sample_files[0]))
print(f"ECG geladen: {sample_files[0].name}")
print(f"  Shape             : {test_ecg.shape}  (verwacht: (5000, 12))")
print(f"  Min/Max (μV)      : {test_ecg.min():.1f} / {test_ecg.max():.1f}")
print(f"  Na normalisatie   : {(test_ecg/ECG_NORM).min():.4f} / {(test_ecg/ECG_NORM).max():.4f}")

## Cel 5 – ECG visualiseren

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(12, 1, figsize=(18, 14), sharex=True)
t = np.linspace(0, 10, ECG_SAMPLES)

for i, (lead, ax) in enumerate(zip(LEAD_ORDER, axes)):
    ax.plot(t, test_ecg[:, i], linewidth=0.6, color="royalblue")
    ax.set_ylabel(lead, fontsize=8, rotation=0, labelpad=25)
    ax.set_yticks([])
    ax.spines[["top","right","left"]].set_visible(False)

axes[-1].set_xlabel("Tijd (s)")
fig.suptitle(f"12-afleidingen ECG – {sample_files[0].stem}", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ecg_voorbeeld.png"), dpi=150)
plt.show()

## Cel 6 – Model laden

In [ ]:
import tensorflow as tf
print(f"TensorFlow versie: {tf.__version__}")

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print(f"\nModel: {MODEL_NAME}")
print(f"  Inputs : {[i.shape for i in model.inputs]}")
print(f"  Outputs: {[o.shape for o in model.outputs]}")

USE_DEMO = len(model.inputs) > 1
HAS_LVH  = len(model.outputs) > 1
print(f"  Demografische input vereist : {USE_DEMO}")
print(f"  LVH-uitvoer beschikbaar     : {HAS_LVH}")

## Cel 7 – Inferentie op alle patiënten

In [ ]:
from tqdm.auto import tqdm

labels_df = pd.read_csv(LABELS_CSV, dtype={"sample_id": str})
labels_df["sample_id"] = labels_df["sample_id"].str.strip()

# Index ECG-bestanden op naam
ecg_index = {Path(p).stem: str(p) for p in Path(ECG_DIR).iterdir()}

records = []
loader  = LOADERS[ECG_FORMAT]

for _, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="Inferentie"):
    sid = str(row["sample_id"])
    if sid not in ecg_index:
        print(f"  ⚠ Geen ECG voor {sid}")
        continue
    try:
        ecg  = loader(ecg_index[sid])
        norm = (ecg / ECG_NORM)[np.newaxis, ...]   # (1, 5000, 12)

        if USE_DEMO:
            demo = np.array([[row["age"], row["sex"], row["bmi"]]], dtype=np.float32)
            out  = model.predict([norm, demo], verbose=0)
        else:
            out  = model.predict(norm, verbose=0)

        rec = {"sample_id": sid}
        if isinstance(out, (list, tuple)):
            rec["lvm_pred"] = float(out[0][0])
            if len(out) > 1:
                rec["lvh_prob"] = float(out[1][0])
        else:
            rec["lvm_pred"] = float(out[0])

        for col in ["lvm_g", "lvh_label", "age", "sex", "bmi"]:
            if col in row.index:
                rec[col] = row[col]
        records.append(rec)
    except Exception as e:
        print(f"  ✗ Fout voor {sid}: {e}")

results = pd.DataFrame(records)
print(f"\nVoorspellingen voor {len(results)} patiënten")
results.head()

## Cel 8 – LV-massa regressie evaluatie

In [ ]:
import seaborn as sns
from scipy import stats

if "lvm_g" in results.columns and results["lvm_g"].notna().sum() > 0:
    sub = results[["lvm_g", "lvm_pred"]].dropna()
    true, pred = sub["lvm_g"].values, sub["lvm_pred"].values

    mae  = np.mean(np.abs(pred - true))
    rmse = np.sqrt(np.mean((pred - true)**2))
    r, p = stats.pearsonr(true, pred)
    slope, intercept, *_ = stats.linregress(true, pred)

    print(f"LVM regressie  (n={len(sub)})")
    print(f"  MAE              : {mae:.1f} g")
    print(f"  RMSE             : {rmse:.1f} g")
    print(f"  Pearson r        : {r:.3f}  (p={p:.3e})")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(true, pred, alpha=0.5, s=20, label="Patiënten")
    lims = [min(true.min(), pred.min())-10, max(true.max(), pred.max())+10]
    ax.plot(lims, lims, "--", color="gray", label="Ideaal")
    x_fit = np.linspace(*lims, 100)
    ax.plot(x_fit, slope*x_fit+intercept, color="red", label=f"Regressie (r={r:.2f})")
    ax.set_xlabel("Gemeten LVM (g)"); ax.set_ylabel("Voorspeld LVM (g)")
    ax.set_title("LVM-AI: Gemeten vs Voorspeld")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "lvm_scatter.png"), dpi=150)
    plt.show()

    # Bland-Altman plot
    mean_val = (true + pred) / 2
    diff     = pred - true
    fig, ax  = plt.subplots(figsize=(6, 4))
    ax.scatter(mean_val, diff, alpha=0.5, s=20)
    ax.axhline(diff.mean(), color="red", label=f"Bias: {diff.mean():.1f} g")
    ax.axhline(diff.mean()+1.96*diff.std(), color="orange", linestyle="--",
               label=f"+1.96 SD: {diff.mean()+1.96*diff.std():.1f} g")
    ax.axhline(diff.mean()-1.96*diff.std(), color="orange", linestyle="--",
               label=f"-1.96 SD: {diff.mean()-1.96*diff.std():.1f} g")
    ax.set_xlabel("Gemiddelde LVM (g)"); ax.set_ylabel("Verschil (voorspeld − gemeten)")
    ax.set_title("Bland-Altman plot")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "bland_altman.png"), dpi=150)
    plt.show()
else:
    print("Geen lvm_g kolom aanwezig – sla regressie-evaluatie over")

## Cel 9 – LVH classificatie evaluatie (c-statistiek / AUC)

In [ ]:
if "lvh_label" in results.columns and "lvh_prob" in results.columns:
    from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, confusion_matrix

    sub  = results[["lvh_label", "lvh_prob"]].dropna()
    y_t  = sub["lvh_label"].astype(int).values
    y_p  = sub["lvh_prob"].values

    auc  = roc_auc_score(y_t, y_p)
    ap   = average_precision_score(y_t, y_p)

    # Bootstrap CI
    rng2 = np.random.default_rng(42)
    ba   = [roc_auc_score(y_t[i:=rng2.integers(0,len(y_t),len(y_t))], y_p[i])
            for _ in range(2000) if len(np.unique(y_t[rng2.integers(0,len(y_t),len(y_t))])) > 1]
    ci_lo, ci_hi = np.percentile(ba, [2.5, 97.5])

    print(f"LVH classificatie  (n={len(sub)}, prevalentie={y_t.mean():.1%})")
    print(f"  AUC-ROC (c-statistiek) : {auc:.4f}  95%CI [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  AUC-PR (gem. precisie) : {ap:.4f}")

    # ROC-curve
    fpr, tpr, thr = roc_curve(y_t, y_p)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(fpr, tpr, label=f"AUC = {auc:.3f} [{ci_lo:.3f}–{ci_hi:.3f}]")
    axes[0].plot([0,1],[0,1],"--",color="gray")
    axes[0].set_xlabel("1 – specificiteit"); axes[0].set_ylabel("Sensitiviteit")
    axes[0].set_title("ROC-curve LVH")
    axes[0].legend()

    # Kalibratie
    from sklearn.calibration import calibration_curve
    prob_true, prob_pred = calibration_curve(y_t, y_p, n_bins=10)
    axes[1].plot(prob_pred, prob_true, marker="o", label="Model")
    axes[1].plot([0,1],[0,1],"--",color="gray",label="Ideaal")
    axes[1].set_xlabel("Voorspelde kans"); axes[1].set_ylabel("Werkelijke frequentie")
    axes[1].set_title("Kalibratie-plot LVH")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_kalibratie.png"), dpi=150)
    plt.show()
else:
    print("lvh_label of lvh_prob kolom ontbreekt – sla LVH-evaluatie over")

## Cel 10 – Resultaten opslaan

In [ ]:
out_csv = os.path.join(OUTPUT_DIR, "predictions.csv")
results.to_csv(out_csv, index=False)
print(f"Voorspellingen opgeslagen: {out_csv}")
print(f"Afbeeldingen opgeslagen in: {OUTPUT_DIR}")
results.describe().round(2)

## Bijlage – XML-formaat en vereisten

### Verwacht XML-formaat (GE MUSE / Philips)

```xml
<RestingECG>
  <PatientDemographics>
    <PatientID>1234567</PatientID>
  </PatientDemographics>
  <Waveform>
    <WaveformType>Rhythm</WaveformType>  <!-- VERPLICHT -->
    <LeadData>
      <LeadID>I</LeadID>
      <LeadSampleSize>2</LeadSampleSize>
      <LeadAmplitudeUnitsPerBit>2.5</LeadAmplitudeUnitsPerBit>
      <LeadSampleCountTotal>5000</LeadSampleCountTotal>
      <WaveformData>BASE64_ENCODED_INT16_DATA</WaveformData>
    </LeadData>
    <!-- Minimaal: I, II, V1-V6 (III/aVR/aVL/aVF worden berekend) -->
  </Waveform>
</RestingECG>
```

### Checklist voor uw data
- [ ] ECG-duur: **10 seconden** (5000 samples bij 500 Hz of 2500 bij 250 Hz)
- [ ] **12 afleiders**: I, II, V1–V6 minimaal (III, aVR, aVL, aVF worden berekend)
- [ ] Bestandsnaam = sample_id in labels.csv (zonder extensie)
- [ ] Spanningseenheid: **μV** (typisch ±5000 μV bereik)
- [ ] LVH-drempel: LVM-index ≥ 115 g/m² (man) of ≥ 95 g/m² (vrouw) op CMR